# Stage 2 — Clean & organize

Input: `data/extracted/{text,table,image}_units.jsonl` from `1_data_ingestion.ipynb` — a lossless but raw extraction.

This notebook turns that into chunking-ready data:

1. Load the three raw streams
2. Drop boilerplate text (headers/footers/page numbers)
3. De-hyphenate + normalize whitespace in text content
4. Validate the shared metadata schema (`doc_id`, `page_number`, `chunk_type`) across all three streams
5. Filter degenerate tables/images (near-empty tables, tiny decorative image crops)
6. Save cleaned streams to `data/cleaned/`

Nothing here does chunking yet — that's Stage 3.

In [1]:
import json
import re
from pathlib import Path

import pandas as pd
import pymupdf

In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
EXTRACTED_DIR = DATA_DIR / "extracted"
CLEANED_DIR = DATA_DIR / "cleaned"
CLEANED_DIR.mkdir(parents=True, exist_ok=True)

MIN_IMAGE_SIDE_PX = 80  # crops smaller than this in width or height are treated as icons/decoration
MIN_TABLE_CELLS = 4     # tables with fewer non-empty cells than this are treated as extraction noise

In [3]:
def load_jsonl(path: Path) -> list[dict]:
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]


text_units = load_jsonl(EXTRACTED_DIR / "text_units.jsonl")
table_units = load_jsonl(EXTRACTED_DIR / "table_units.jsonl")
image_units = load_jsonl(EXTRACTED_DIR / "image_units.jsonl")

print(f"raw text_units:  {len(text_units)}")
print(f"raw table_units: {len(table_units)}")
print(f"raw image_units: {len(image_units)}")

raw text_units:  10606
raw table_units: 25
raw image_units: 73


## Drop boilerplate

Stage 1 flagged `is_boilerplate` for `pageHeader`/`pageFooter`/`pageNumber` roles but kept them, so this notebook could decide independently. Repeated headers/footers add no information and would show up as near-duplicate chunks across every page — drop them here.

In [4]:
n_before = len(text_units)
boilerplate_examples = [u["content"] for u in text_units if u.get("is_boilerplate")][:10]

text_units = [u for u in text_units if not u.get("is_boilerplate")]

print(f"dropped {n_before - len(text_units)} boilerplate units, {len(text_units)} remain")
print("examples of what was dropped:")
for ex in boilerplate_examples:
    print(" -", ex)

dropped 1043 boilerplate units, 9563 remain
examples of what was dropped:
 - 923
 - Chapter 7
 - The Earth's Energy Budget, Climate Feedbacks and Climate Sensitivity
 - 924
 - 7
 - The Earth's Energy Budget, Climate Feedbacks and Climate Sensitivity
 - Chapter 7
 - 925
 - 7
 - Chapter 7


## De-hyphenate + normalize whitespace

PDF line-wrap hyphenation shows up as `word- continuation` (hyphen directly followed by whitespace then the rest of the word) — genuine hyphenated compounds like `well-being` never have a space after the hyphen, so matching only `-<whitespace>` is a safe heuristic. Spot-check a few examples below before trusting it on the full set.

In [5]:
HYPHEN_BREAK_RE = re.compile(r"(\w)-\s+(\w)")
WHITESPACE_RE = re.compile(r"\s+")

_sample_matches = []
for u in text_units:
    m = HYPHEN_BREAK_RE.search(u["content"])
    if m:
        start, end = max(0, m.start() - 20), min(len(u["content"]), m.end() + 20)
        _sample_matches.append(u["content"][start:end])
    if len(_sample_matches) >= 10:
        break

print(f"paragraphs with a hyphen-break match (showing up to 10): {len(_sample_matches)}")
for ex in _sample_matches:
    print(" -", repr(ex))

paragraphs with a hyphen-break match (showing up to 10): 10
 - ' or by treating short- and long-lived GHG em'
 - 'lications of short-li- ved and long-lived cl'
 - 'Dutton, 2013; Sanchez- Lorenzo et al., 2015;'
 - 'tospheric-temperature- adjusted radiative fo'
 - 'Stratospheric- temperature- adjusted'
 - 'Stratospheric- temperature- adjusted'
 - 'al effects (Doutriaux- Boucher et al., 2009;'
 - ' affected cloud micro- and macro-physics and'
 - 'cusing on observation- based (Section 7.3.3.'
 - 'ormed new observation- and model-based estim'


In [6]:
def clean_text(content: str) -> str:
    content = HYPHEN_BREAK_RE.sub(r"\1\2", content)
    content = WHITESPACE_RE.sub(" ", content)
    return content.strip()


for unit in text_units:
    unit["content"] = clean_text(unit["content"])

pd.DataFrame(text_units).sample(5)

,doc_id,page_number,chunk_type,role,is_boilerplate,content,bbox,raw_ref
2827,IPCC_AR6_WGI_Chapter07.pdf,71,text,NaN,False,"Before AR6, the assessment of ECS relied on ei...","[0.7712, 2.0815, 4.2116, 2.083, 4.21, 5.907, 0...",3104
8014,IPCC_AR6_WGI_Chapter08.pdf,116,text,NaN,False,"Descroix, L. et al., 2013: Évolution des pluie...","[0.6992, 4.6558, 4.126, 4.6586, 4.1257, 5.1094...",3644
2685,IPCC_AR6_WGI_Chapter07.pdf,64,text,NaN,False,90N,"[2.907, 6.7131, 3.0551, 6.7137, 3.0548, 6.7934...",2934
5995,IPCC_AR6_WGI_Chapter08.pdf,48,text,NaN,False,"(e) Northern Hemisphere, JFM, boreal winter","[1.7552, 5.7639, 3.882, 5.7594, 3.8823, 5.92, ...",1391
5511,IPCC_AR6_WGI_Chapter08.pdf,34,text,NaN,False,-0.004,"[2.4247, 7.1407, 2.6869, 7.1403, 2.6871, 7.240...",859


## Filter degenerate tables

A DI "table" detection is occasionally just noise — a near-empty grid or a single cell. Drop tables with fewer than `MIN_TABLE_CELLS` non-empty cells; count is taken directly from the rendered Markdown row/pipe structure rather than re-parsing the original DI cell list, since the Markdown is what actually gets embedded.

In [7]:
def count_nonempty_cells(table_markdown: str) -> int:
    lines = table_markdown.split("\n")
    body_lines = [lines[0]] + lines[2:]  # skip the "| --- | --- |" separator row
    return sum(1 for line in body_lines for cell in line.strip("|").split("|") if cell.strip())


n_before = len(table_units)
table_units = [u for u in table_units if count_nonempty_cells(u["content"]) >= MIN_TABLE_CELLS]
print(f"dropped {n_before - len(table_units)} degenerate tables, {len(table_units)} remain")

dropped 0 degenerate tables, 25 remain


## Filter tiny image crops

Some DI "figures" are decorative icons/logos rather than real charts or diagrams. Read each saved crop's pixel dimensions with `pymupdf.Pixmap` (no new dependency needed) and drop anything smaller than `MIN_IMAGE_SIDE_PX` on either side.

In [8]:
def is_large_enough(image_path: str, min_side: int = MIN_IMAGE_SIDE_PX) -> bool:
    pix = pymupdf.Pixmap(str(PROJECT_ROOT / image_path))
    return pix.width >= min_side and pix.height >= min_side


n_before = len(image_units)
image_units = [u for u in image_units if is_large_enough(u["image_path"])]
print(f"dropped {n_before - len(image_units)} tiny image crops, {len(image_units)} remain")

dropped 2 tiny image crops, 71 remain


## Validate the shared metadata schema

`doc_id`, `page_number`, and `chunk_type` must be present and non-null on every unit across all three streams — these are what the DOCX export cites at the very end of the pipeline, so a gap here surfaces now instead of as a broken citation much later.

In [9]:
REQUIRED_FIELDS = ["doc_id", "page_number", "chunk_type"]  # image content is legitimately None until Stage 3 captioning


def validate_units(units: list[dict], stream_name: str) -> None:
    bad = [
        (i, field)
        for i, u in enumerate(units)
        for field in REQUIRED_FIELDS
        if u.get(field) in (None, "")
    ]
    if bad:
        print(f"{stream_name}: {len(bad)} field violations, e.g. {bad[:5]}")
    else:
        print(f"{stream_name}: OK ({len(units)} units, all required fields present)")


validate_units(text_units, "text_units")
validate_units(table_units, "table_units")
validate_units(image_units, "image_units")

text_units: OK (9563 units, all required fields present)
table_units: OK (25 units, all required fields present)
image_units: OK (71 units, all required fields present)


## Save cleaned streams

Written to `data/cleaned/` — kept separate from `data/extracted/` so Stage 1's raw output stays an untouched fallback.

In [10]:
def save_jsonl(units: list[dict], path: Path) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for unit in units:
            f.write(json.dumps(unit) + "\n")


save_jsonl(text_units, CLEANED_DIR / "text_units.jsonl")
save_jsonl(table_units, CLEANED_DIR / "table_units.jsonl")
save_jsonl(image_units, CLEANED_DIR / "image_units.jsonl")

print(f"cleaned text_units:  {len(text_units)}")
print(f"cleaned table_units: {len(table_units)}")
print(f"cleaned image_units: {len(image_units)}")

cleaned text_units:  9563
cleaned table_units: 25
cleaned image_units: 71


## Next: Stage 3 — Semantic chunking + embeddings

Reads `data/cleaned/*.jsonl`. Text gets chunked with LangChain's `SemanticChunker`; tables stay one-chunk-per-table as-is; each image gets captioned by a vision LLM (Azure OpenAI GPT-4o) to produce its `content` before anything gets embedded.